# MiniGPT · a GPT in about 100 lines

We build the smallest model that still contains **every essential idea of a GPT**, train it on Shakespeare and watch it learn to write.

```
text → tokens → embeddings → [ attention + MLP ] × N → logits → next token
```

Shapes: `B` = batch, `T` = tokens per sequence, `C` = vector size per token.

---
# Part 1 · Building the model

## 1 · Data and tokenizer
- We download ~1M characters of Shakespeare.
- **Tokenizer:** every distinct character gets an ID (65 in total). `encode` turns text into IDs, `decode` turns IDs back into text. The model never sees text, only IDs.
- 90% of the IDs are for **training**, 10% for **validation** (text the model never trains on).

In [ ]:
import os, urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt", encoding="utf-8").read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(device, "| vocab_size:", vocab_size, "| tokens:", len(data))

## 2 · Settings and batches
- `block_size` is the **context window**: how many tokens the model can look at.
- `get_batch` picks `batch_size` random chunks. `y` is **the same chunk shifted by one**, so at every position the target is simply the next character. No labels needed: the text labels itself.

In [ ]:
block_size = 64      # context window
batch_size = 32
n_embd     = 128     # vector size per token
n_layer    = 4       # number of blocks
max_iters  = 2000
lr         = 1e-3

def get_batch(d):
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])   # shifted by one
    return x.to(device), y.to(device)

## 3 · Self-attention: how tokens talk to each other
Each token makes three vectors:
- **query**: "what am I looking for?"
- **key**: "what do I contain?"
- **value**: "what do I hand over if chosen?"

Then:
1. `q @ kᵀ` scores how well every token matches every other token → a `(T, T)` table.
2. Divide by $\sqrt{C}$ so the scores don't get too large.
3. **Causal mask:** a token may only look at itself and the past. Future positions get $-\infty$ → weight 0. Otherwise the model would see the answer it has to predict.
4. `softmax` turns each row into weights that sum to 1.
5. Each token's output = a weighted average of the values it attends to.

$$\text{softmax}\!\left(\frac{QK^T}{\sqrt{C}} + \text{mask}\right)V$$

*(Real GPTs run several of these in parallel, "multi-head attention", each looking for a different kind of relationship. The idea is exactly the same, so we keep one.)*

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.key   = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):                                    # x: (B, T, C)
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / C ** 0.5          # (B, T, T)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return weights @ v                                   # (B, T, C)

## 4 · The Transformer block
One block = **communicate** (attention: tokens exchange information) + **think** (MLP: each token processes what it gathered, on its own).

Two tricks make deep stacks trainable:
- **Residual connection** `x = x + ...`: each part only *adds* a correction to `x`, and gradients flow straight back through the `+`.
- **LayerNorm**: keeps the numbers in each token's vector at a stable scale.

Input `(B, T, C)` → output `(B, T, C)`, so blocks can be stacked as many times as we like.

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = SelfAttention()
        self.ln2  = nn.LayerNorm(n_embd)
        self.mlp  = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))    # communicate
        x = x + self.mlp(self.ln2(x))     # think
        return x

## 5 · The full model
1. **Token embedding:** a learnable vector for each token ID, *what* the token is.
2. **Position embedding:** a learnable vector for each position, *where* it is. Attention alone has no sense of order, so we add the two.
3. **N blocks.**
4. **Final layer:** turns each vector into **one score (logit) per vocabulary character**, "how likely is each one to come next".
5. **Loss:** cross-entropy, $-\ln(p_{\text{correct next char}})$, averaged over all positions.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.head    = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):                    # idx: (B, T)
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.blocks(x)
        logits = self.head(self.ln_f(x))                     # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

## 6 · Generation
Exactly the loop from Session 1: **predict → pick → append → repeat**.
- `idx[:, -block_size:]`: the model only sees the last `block_size` tokens (the context window).
- `logits[:, -1]`: we only need the prediction at the last position.
- **temperature** divides the logits before softmax: low = safe and repetitive, high = creative and error-prone.

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=200, temperature=1.0):
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return decode(idx[0].tolist())

## 7 · Before training
- An untrained model gives every character about the same probability, $1/65$, so the loss should be about $\ln 65 \approx 4.17$.
- It already generates text end to end, but it is random characters: **the architecture is there, the knowledge is not**.

In [ ]:
model = MiniGPT().to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

_, loss = model(*get_batch(train_data))
print(f"loss: {loss.item():.2f}  (random guessing ≈ 4.17)")
print(generate("ROMEO:", 100))

---
# Part 2 · Training and generation

## 8 · Training
**The same 4 steps as `TinyModel` in Session 2:** forward → loss → backward → step. Only the model is bigger and the loss is cross-entropy.

We print the training loss and the loss on **validation** text the model never trains on. If both fall, the model is really learning the language, not memorising. Expect it to go from ~4.2 down to ~1.7.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(max_iters + 1):
    xb, yb = get_batch(train_data)
    _, loss = model(xb, yb)          # 1. forward + 2. loss
    optimizer.zero_grad()
    loss.backward()                  # 3. backward
    optimizer.step()                 # 4. step

    if step % 200 == 0:
        with torch.no_grad():
            _, val_loss = model(*get_batch(val_data))
        print(f"step {step:4d} | train {loss.item():.2f} | val {val_loss.item():.2f}")

## 9 · After training
Same prompt, same `generate`. Now it writes like a play: speaker names, line breaks and mostly real English words. Compare the two temperatures.

In [ ]:
print(generate("ROMEO:", temperature=0.8))
print("\n" + "-" * 40 + "\n")
print(generate("ROMEO:", temperature=1.5))